# DynamoDB

## ML Inference Logging (Object Recognition)

Object recognition models (YOLO, Rekognition, etc.) produce a new record for every image or video frame processed — potentially millions per day. DynamoDB is well-suited because:

| Requirement | Why DynamoDB fits |
|-------------|------------------|
| **High write throughput** | Every inference is a write; DynamoDB scales horizontally with no bottleneck |
| **Variable output schema** | Model v1 may return bounding boxes; v2 adds segmentation masks. Schemaless items absorb this without `ALTER TABLE` |
| **Time-ordered review** | SK = `timestamp#id` gives chronological ordering per image/camera for free |
| **Auto-expiry (TTL)** | One `ttl` field auto-deletes stale logs after days — no cron jobs |
| **Fast single-image lookup** | `get_item(imageId)` is O(1) — instant retrieval of a specific image's results |
| **Low write latency** | Single-digit millisecond writes don't block the inference pipeline |

### What we are NOT storing in DynamoDB
- The **actual image bytes** → stay in S3; we store only the S3 URI
- Large embedding vectors → store in a vector DB (e.g., OpenSearch, pgvector)
- Aggregated analytics → use DynamoDB Streams → S3 → Athena for that

## Table Design

```
Table: MLInferences
  Partition Key (PK):  imageId      (S)  ← identifies the source image / camera frame
  Sort Key      (SK):  inferenceId  (S)  ← "{ISO-timestamp}#{uuid4}"  (time-ordered per image)
```

Each item stores the full detection payload for one model run against one image:

```json
{
  "imageId":          "cam01_20260410_103045",
  "inferenceId":      "2026-04-10T10:30:45Z#a3f8c2d1",
  "modelName":        "yolo-v8",
  "modelVersion":     "1.4.0",
  "processingTimeMs": 38,
  "s3Uri":            "s3://my-bucket/frames/cam01_20260410_103045.jpg",
  "detections": [
    { "class": "car",    "confidence": 0.94, "bbox": {"x": 120, "y": 80,  "w": 310, "h": 140} },
    { "class": "person", "confidence": 0.87, "bbox": {"x": 450, "y": 200, "w": 60,  "h": 190} }
  ],
  "detectionCount":   2,
  "ttl":              1776700245
}
```

https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/dynamodb.html

## Imports

In [1]:
import boto3
import json
import random
import uuid
import time
import pandas as pd
from decimal import Decimal
from datetime import datetime, timezone, timedelta
from boto3.dynamodb.conditions import Key, Attr

## AWS Connection

In [2]:
session = boto3.Session(profile_name='ashish')
credentials = session.get_credentials()

aws_access_key_id     = credentials.access_key
aws_secret_access_key = credentials.secret_key
region_name           = 'us-east-1'

In [3]:
dynamodb = boto3.client(
    'dynamodb',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name,
)

dynamodb_resource = boto3.resource(
    'dynamodb',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name,
)

print('DynamoDB client and resource ready.')

DynamoDB client and resource ready.


## Create Table – MLInferences

Only the **key attributes** are declared in advance. All other inference fields (detections, S3 URI, model version, TTL) are schemaless — the table will accept any extra attributes without a schema change.

In [4]:
TABLE_NAME = 'MLInferences'
TTL_DAYS   = 30   # inferences older than this are automatically deleted

existing = dynamodb.list_tables()['TableNames']
if TABLE_NAME in existing:
    print(f"Table '{TABLE_NAME}' already exists – skipping creation.")
else:
    dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[
            {'AttributeName': 'imageId',     'KeyType': 'HASH'},
            {'AttributeName': 'inferenceId', 'KeyType': 'RANGE'},
        ],
        AttributeDefinitions=[
            {'AttributeName': 'imageId',     'AttributeType': 'S'},
            {'AttributeName': 'inferenceId', 'AttributeType': 'S'},
        ],
        BillingMode='PAY_PER_REQUEST',
    )
    waiter = dynamodb.get_waiter('table_exists')
    waiter.wait(TableName=TABLE_NAME)
    print(f"Table '{TABLE_NAME}' created.")

# Enable TTL on the 'ttl' attribute
dynamodb.update_time_to_live(
    TableName=TABLE_NAME,
    TimeToLiveSpecification={'Enabled': True, 'AttributeName': 'ttl'},
)
print(f"TTL enabled on attribute 'ttl' (items expire after {TTL_DAYS} days).")

tbl = dynamodb_resource.Table(TABLE_NAME)

Table 'MLInferences' created.
TTL enabled on attribute 'ttl' (items expire after 30 days).


## Simulate ML Model Output

We simulate the output of an object-recognition model (e.g., YOLO). Each call to `simulate_inference()` mimics what the model would return for one image:

- A variable number of detected objects, each with a **class label**, **confidence score**, and **bounding box**
- Processing time in milliseconds
- A reference to the source image in S3

In production this function would be replaced by a real model call (`torch`, AWS Rekognition, etc.).

In [5]:
# Classes our simulated object detector can recognise
OBJECT_CLASSES = ['car', 'person', 'bicycle', 'truck', 'dog', 'traffic_light', 'bus', 'motorcycle']
CAMERAS        = ['cam01', 'cam02', 'cam03', 'cam04']
MODEL_NAME     = 'yolo-v8'
MODEL_VERSION  = '1.4.0'
IMAGE_W, IMAGE_H = 1920, 1080


def simulate_inference(image_id: str, s3_uri: str) -> dict:
    """
    Simulate the output of an object-recognition model for a single image.
    Returns a dict ready to be written to DynamoDB.
    """
    num_detections = random.randint(0, 6)
    detections = []
    for _ in range(num_detections):
        x = random.randint(0, IMAGE_W - 100)
        y = random.randint(0, IMAGE_H - 100)
        w = random.randint(40, min(400, IMAGE_W - x))
        h = random.randint(40, min(300, IMAGE_H - y))
        detections.append({
            'class':      random.choice(OBJECT_CLASSES),
            'confidence': round(random.uniform(0.50, 0.99), 3),
            'bbox':       {'x': x, 'y': y, 'w': w, 'h': h},
        })

    ts  = datetime.now(timezone.utc)
    ttl = int((ts + timedelta(days=TTL_DAYS)).timestamp())

    return {
        'imageId':          image_id,
        'inferenceId':      f"{ts.strftime('%Y-%m-%dT%H:%M:%SZ')}#{uuid.uuid4().hex[:8]}",
        'modelName':        MODEL_NAME,
        'modelVersion':     MODEL_VERSION,
        'timestamp':        ts.isoformat(),
        'processingTimeMs': random.randint(20, 120),
        's3Uri':            s3_uri,
        'imageWidth':       IMAGE_W,
        'imageHeight':      IMAGE_H,
        'detections':       detections,
        'detectionCount':   num_detections,
        'ttl':              ttl,
    }


# Preview one simulated result before writing anything
sample = simulate_inference('cam01_preview', 's3://my-bucket/frames/cam01_preview.jpg')
print(json.dumps(sample, indent=2, default=str))

{
  "imageId": "cam01_preview",
  "inferenceId": "2026-04-10T20:20:48Z#983d4743",
  "modelName": "yolo-v8",
  "modelVersion": "1.4.0",
  "timestamp": "2026-04-10T20:20:48.676399+00:00",
  "processingTimeMs": 119,
  "s3Uri": "s3://my-bucket/frames/cam01_preview.jpg",
  "imageWidth": 1920,
  "imageHeight": 1080,
  "detections": [
    {
      "class": "person",
      "confidence": 0.582,
      "bbox": {
        "x": 901,
        "y": 764,
        "w": 157,
        "h": 121
      }
    },
    {
      "class": "dog",
      "confidence": 0.715,
      "bbox": {
        "x": 1020,
        "y": 808,
        "w": 332,
        "h": 260
      }
    },
    {
      "class": "bicycle",
      "confidence": 0.86,
      "bbox": {
        "x": 949,
        "y": 773,
        "w": 269,
        "h": 202
      }
    },
    {
      "class": "person",
      "confidence": 0.8,
      "bbox": {
        "x": 740,
        "y": 665,
        "w": 55,
        "h": 202
      }
    }
  ],
  "detectionCount": 4,
  "ttl":

## Write Simulated Inferences to DynamoDB

Simulate a batch of 50 inferences spread across 4 cameras. In a production pipeline this loop would be replaced by the actual inference service writing results as images arrive.

DynamoDB's `batch_writer()` automatically groups items into 25-item batch requests (the API maximum), retrying any unprocessed items.

In [6]:
def to_decimal(obj):
    """Recursively convert float → Decimal for DynamoDB compatibility."""
    if isinstance(obj, float):
        return Decimal(str(obj))
    if isinstance(obj, dict):
        return {k: to_decimal(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_decimal(i) for i in obj]
    return obj


NUM_IMAGES = 50
records    = []

for i in range(NUM_IMAGES):
    cam      = random.choice(CAMERAS)
    frame_ts = datetime.now(timezone.utc) - timedelta(minutes=random.randint(0, 120))
    image_id = f"{cam}_{frame_ts.strftime('%Y%m%d_%H%M%S')}_{i:03d}"
    s3_uri   = f"s3://my-inference-bucket/frames/{cam}/{image_id}.jpg"
    records.append(to_decimal(simulate_inference(image_id, s3_uri)))

with tbl.batch_writer() as batch:
    for record in records:
        batch.put_item(Item=record)

print(f"Written {len(records)} inference records to '{TABLE_NAME}'.")
print(f"Total detections across all records: {sum(r['detectionCount'] for r in records)}")

Written 50 inference records to 'MLInferences'.
Total detections across all records: 164


## Query Pattern – All inferences for a specific image

The most common review query: *"What did the model detect every time it processed image X?"*

Because `imageId` is the partition key, this is a cheap key-only read — DynamoDB reads only one partition, regardless of total table size.

In [7]:
# Pick the first image we wrote so we know it exists
target_image = records[0]['imageId']

response = tbl.query(
    KeyConditionExpression=Key('imageId').eq(target_image)
)

items = response['Items']
print(f"Image '{target_image}' — {len(items)} inference(s) found:")
for item in items:
    print(f"  [{item['timestamp']}]  {item['detectionCount']} object(s) detected  "
          f"| model={item['modelName']} v{item['modelVersion']}  "
          f"| {item['processingTimeMs']} ms")
    for d in item['detections']:
        print(f"      {d['class']:15s}  conf={float(d['confidence']):.3f}  "
              f"bbox=({d['bbox']['x']},{d['bbox']['y']},{d['bbox']['w']},{d['bbox']['h']})")

Image 'cam01_20260410_193548_000' — 1 inference(s) found:
  [2026-04-10T20:20:48.683763+00:00]  3 object(s) detected  | model=yolo-v8 v1.4.0  | 36 ms
      traffic_light    conf=0.730  bbox=(979,160,54,251)
      bicycle          conf=0.686  bbox=(1408,324,237,258)
      bus              conf=0.744  bbox=(532,747,169,149)


## Query Pattern – Review inferences with low-confidence detections

*"Find all inference records where at least one detection had confidence below a threshold"* — useful for flagging images that need human review.

Because `detectionCount` and `detections` are not key attributes, we must use a `scan()` with a `FilterExpression`. For a high-volume production table you would add a **Global Secondary Index** on a `minConfidence` attribute (pre-computed at write time) to avoid a full scan.

In [8]:
CONFIDENCE_THRESHOLD = Decimal('0.6')

# Compute minConfidence at query time from the detections list
# (In production, store minConfidence as a top-level attribute at write time
#  so you can index it with a GSI instead of scanning.)
paginator = dynamodb.get_paginator('scan')
low_conf_items = []

for page in paginator.paginate(TableName=TABLE_NAME):
    for raw in page['Items']:
        # Deserialize typed values: {'N': '0.75'} → Decimal('0.75'), {'S': 'car'} → 'car'
        item = {k: (Decimal(v['N']) if 'N' in v else list(v.values())[0])
                for k, v in raw.items()}
        # Re-fetch via resource for nested list deserialization
        pass

# Use resource-level scan for automatic deserialization of nested dicts/lists
low_conf_items = []
last_key = None
while True:
    kwargs = {'FilterExpression': Attr('detectionCount').gt(0)}
    if last_key:
        kwargs['ExclusiveStartKey'] = last_key
    resp = tbl.scan(**kwargs)
    for item in resp['Items']:
        confidences = [d['confidence'] for d in item.get('detections', [])]
        if confidences and min(confidences) < CONFIDENCE_THRESHOLD:
            low_conf_items.append(item)
    last_key = resp.get('LastEvaluatedKey')
    if not last_key:
        break

print(f"Records with at least one detection below {float(CONFIDENCE_THRESHOLD):.2f} confidence: {len(low_conf_items)}")

# Show a summary DataFrame
rows = []
for item in low_conf_items:
    for d in item['detections']:
        if d['confidence'] < CONFIDENCE_THRESHOLD:
            rows.append({
                'imageId':    item['imageId'],
                'timestamp':  item['timestamp'],
                'class':      d['class'],
                'confidence': float(d['confidence']),
                's3Uri':      item['s3Uri'],
            })

df_review = pd.DataFrame(rows).sort_values('confidence')
print(f"Low-confidence detections to review: {len(df_review)}")
df_review.head(10)

Records with at least one detection below 0.60 confidence: 23
Low-confidence detections to review: 34


,imageId,timestamp,class,confidence,s3Uri
20,cam04_20260410_201748_044,2026-04-10T20:20:48.687301+00:00,car,0.501,s3://my-inference-bucket/frames/cam04/cam04_20...
10,cam04_20260410_193848_001,2026-04-10T20:20:48.683968+00:00,bicycle,0.509,s3://my-inference-bucket/frames/cam04/cam04_20...
13,cam01_20260410_193348_028,2026-04-10T20:20:48.686302+00:00,bus,0.510,s3://my-inference-bucket/frames/cam01/cam01_20...
0,cam01_20260410_195548_016,2026-04-10T20:20:48.685383+00:00,bus,0.516,s3://my-inference-bucket/frames/cam01/cam01_20...
6,cam02_20260410_200748_035,2026-04-10T20:20:48.686667+00:00,traffic_light,0.516,s3://my-inference-bucket/frames/cam02/cam02_20...
27,cam01_20260410_184648_017,2026-04-10T20:20:48.685439+00:00,car,0.517,s3://my-inference-bucket/frames/cam01/cam01_20...
26,cam02_20260410_191948_010,2026-04-10T20:20:48.684898+00:00,dog,0.518,s3://my-inference-bucket/frames/cam02/cam02_20...
16,cam02_20260410_193748_018,2026-04-10T20:20:48.685514+00:00,traffic_light,0.525,s3://my-inference-bucket/frames/cam02/cam02_20...
17,cam02_20260410_193748_018,2026-04-10T20:20:48.685514+00:00,truck,0.527,s3://my-inference-bucket/frames/cam02/cam02_20...
8,cam02_20260410_183048_014,2026-04-10T20:20:48.685257+00:00,truck,0.531,s3://my-inference-bucket/frames/cam02/cam02_20...


## Query Pattern – All detections of a specific class (e.g., 'person')

*"Show me every inference that detected a person across all cameras."*

This again requires a `scan()` because `class` is inside a nested list — not a top-level attribute. The production fix is to **denormalize**: store a `detectedClasses` string set (`SS`) at the top level so DynamoDB can use `contains` in a FilterExpression more efficiently, or add a GSI.

In [9]:
TARGET_CLASS = 'person'

class_hits = []
last_key   = None
while True:
    kwargs = {}
    if last_key:
        kwargs['ExclusiveStartKey'] = last_key
    resp = tbl.scan(**kwargs)
    for item in resp['Items']:
        matched = [d for d in item.get('detections', []) if d['class'] == TARGET_CLASS]
        if matched:
            class_hits.append({
                'imageId':      item['imageId'],
                'timestamp':    item['timestamp'],
                'detections':   matched,
                's3Uri':        item['s3Uri'],
                'processingMs': item['processingTimeMs'],
            })
    last_key = resp.get('LastEvaluatedKey')
    if not last_key:
        break

print(f"Images containing a '{TARGET_CLASS}': {len(class_hits)}")

rows = []
for hit in class_hits:
    for d in hit['detections']:
        rows.append({
            'imageId':    hit['imageId'],
            'timestamp':  hit['timestamp'],
            'confidence': float(d['confidence']),
            'bbox_x':     int(d['bbox']['x']),
            'bbox_y':     int(d['bbox']['y']),
            'bbox_w':     int(d['bbox']['w']),
            'bbox_h':     int(d['bbox']['h']),
            's3Uri':      hit['s3Uri'],
        })

df_class = pd.DataFrame(rows).sort_values('confidence', ascending=False)
df_class.head(10)

Images containing a 'person': 17


,imageId,timestamp,confidence,bbox_x,bbox_y,bbox_w,bbox_h,s3Uri
0,cam03_20260410_201848_024,2026-04-10T20:20:48.686005+00:00,0.981,1153,686,140,176,s3://my-inference-bucket/frames/cam03/cam03_20...
20,cam03_20260410_183248_019,2026-04-10T20:20:48.685606+00:00,0.959,1549,371,264,271,s3://my-inference-bucket/frames/cam03/cam03_20...
9,cam04_20260410_194648_038,2026-04-10T20:20:48.686887+00:00,0.944,1367,659,236,270,s3://my-inference-bucket/frames/cam04/cam04_20...
8,cam02_20260410_183048_014,2026-04-10T20:20:48.685257+00:00,0.899,591,936,53,62,s3://my-inference-bucket/frames/cam02/cam02_20...
15,cam03_20260410_200448_027,2026-04-10T20:20:48.686222+00:00,0.870,113,185,184,48,s3://my-inference-bucket/frames/cam03/cam03_20...
18,cam04_20260410_194548_013,2026-04-10T20:20:48.685158+00:00,0.868,1388,845,206,61,s3://my-inference-bucket/frames/cam04/cam04_20...
19,cam03_20260410_183248_019,2026-04-10T20:20:48.685606+00:00,0.862,1633,224,123,202,s3://my-inference-bucket/frames/cam03/cam03_20...
17,cam02_20260410_191948_010,2026-04-10T20:20:48.684898+00:00,0.837,1309,651,286,136,s3://my-inference-bucket/frames/cam02/cam02_20...
5,cam02_20260410_200748_035,2026-04-10T20:20:48.686667+00:00,0.761,222,443,193,105,s3://my-inference-bucket/frames/cam02/cam02_20...
3,cam01_20260410_183348_008,2026-04-10T20:20:48.684717+00:00,0.708,1109,973,332,96,s3://my-inference-bucket/frames/cam01/cam01_20...


## Query Pattern – Inference summary statistics

Because DynamoDB has no server-side aggregation, summary stats are computed client-side in pandas after a full scan. This is fine for periodic review dashboards; for real-time metrics, pipe results via DynamoDB Streams → Lambda → CloudWatch.

In [10]:
# Flatten all detections into a single DataFrame
all_records = []
last_key    = None
while True:
    kwargs = {}
    if last_key:
        kwargs['ExclusiveStartKey'] = last_key
    resp = tbl.scan(**kwargs)
    all_records.extend(resp['Items'])
    last_key = resp.get('LastEvaluatedKey')
    if not last_key:
        break

flat_detections = []
for item in all_records:
    for d in item.get('detections', []):
        flat_detections.append({
            'class':          d['class'],
            'confidence':     float(d['confidence']),
            'processingMs':   int(item['processingTimeMs']),
            'modelVersion':   item['modelVersion'],
        })

df_det = pd.DataFrame(flat_detections)

print("=== Detection counts by class ===")
print(df_det['class'].value_counts().to_string())

print("\n=== Mean confidence by class ===")
print(df_det.groupby('class')['confidence'].mean().round(3).sort_values(ascending=False).to_string())

print(f"\n=== Model processing time ===")
print(f"  Mean:   {df_det['processingMs'].mean():.1f} ms")
print(f"  Median: {df_det['processingMs'].median():.1f} ms")
print(f"  P95:    {df_det['processingMs'].quantile(0.95):.1f} ms")

=== Detection counts by class ===
class
bus              25
traffic_light    25
truck            23
person           23
bicycle          20
car              17
dog              16
motorcycle       15

=== Mean confidence by class ===
class
motorcycle       0.770
bus              0.759
dog              0.748
truck            0.744
bicycle          0.742
traffic_light    0.742
car              0.739
person           0.720

=== Model processing time ===
  Mean:   71.6 ms
  Median: 71.0 ms
  P95:    114.0 ms


## TTL – Why it matters for inference logs

Every item was written with a `ttl` attribute — a Unix epoch timestamp set to `now + 30 days`. DynamoDB's TTL daemon periodically scans for expired items and deletes them automatically at no charge.

This means:
- Inference logs are automatically pruned — the table never grows unboundedly
- No application code or cron job needed to manage retention
- The window can be per-item (e.g., high-value detections keep longer TTLs)

The cell below shows the TTL value on a sample item and how much remaining life it has.

In [11]:
sample_item = all_records[0]
ttl_epoch   = int(sample_item['ttl'])
ttl_dt      = datetime.fromtimestamp(ttl_epoch, tz=timezone.utc)
remaining   = ttl_dt - datetime.now(timezone.utc)

print(f"imageId   : {sample_item['imageId']}")
print(f"ttl (epoch): {ttl_epoch}")
print(f"Expires   : {ttl_dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Remaining : {remaining.days} days, {remaining.seconds // 3600} hours")

imageId   : cam03_20260410_201848_024
ttl (epoch): 1778444448
Expires   : 2026-05-10 20:20:48 UTC
Remaining : 29 days, 23 hours


## Production Recommendations

| Gap in this demo | Production fix |
|-----------------|----------------|
| Filtering by `class` requires a full scan | Add a top-level `detectedClasses` string-set attribute; add a GSI with it as PK |
| Filtering by `confidence` requires a full scan | Store `minConfidence` as a top-level Number attribute; sort by it in a GSI |
| Analytics (counts, averages) require a scan | DynamoDB Streams → Lambda → S3 → Athena for batch analytics |
| Querying across cameras simultaneously | Add a GSI with `cameraId` (extracted from `imageId`) as the partition key |
| Image storage | Store raw frames in S3; keep only the S3 URI in DynamoDB (already done here) |

## Clean Up

Uncomment and run to delete the table and avoid ongoing costs.

In [12]:
# dynamodb.delete_table(TableName=TABLE_NAME)
# waiter = dynamodb.get_waiter('table_not_exists')
# waiter.wait(TableName=TABLE_NAME)
# print(f"Table '{TABLE_NAME}' deleted.")